## Planner Agent

#### Importing libs

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

import os
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))

from agents.research_agent import research_agent
from utils.load_prompt import load_prompt
from utils.render import render_block

load_dotenv()
CLIENT = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL = "gemini-2.5-flash"
USER_COLOR = "#D6EAF8"
AGENT_COLOR = "#F9E79F"

#### Defining auxiliary functions

In [3]:
def pricing_agent(target_neighborhood: str, model: str) -> str:
    """
    Runs the real estate pricing agent.

    The agent estimates a property price for a given neighborhood
    using a predictive model.

    Args:
        target_neighborhood (str): Name of the neighborhood for which to estimate property prices.
        model (str): The model identifier to use for generating pricing predictions.

    Returns:
        str: Estimated pricing for the specified neighborhood.
    """
    return f"""Infelizmente estamos com o serviço fora do ar, não conseguiremos estimar o preço para **{neighborhood}**."""

#### Prompt Engineering

In [ ]:
def planner_agent(model: str) -> None:
    """
    Planner Agent: autonomously decides which agent to call based on user input.

    The agent receives a user prompt and a list of available agent functions,
    and uses the model to determine which agent to invoke, calling it automatically.

    Args:
        model (str): The model identifier to use for planning and decision-making.

    Returns:
        None: Displays the agent's response in markdown.
    """
    CLIENT = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    system_prompt = load_prompt("planner_agent_v1")

    research_agent_def = types.FunctionDeclaration.from_callable(client = CLIENT, callable = research_agent)
    pricing_agent_def = types.FunctionDeclaration.from_callable(client = CLIENT, callable = pricing_agent)
    tools = [research_agent_def, pricing_agent_def]
    
    chat = CLIENT.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            tools=tools,
            temperature=0.7
        )
    )

    while True:
        user_input = input("USER: ")
        if user_input.lower() in ["exit"]:
            break

        render_block("USER", user_input, USER_COLOR)
        resp = chat.send_message(user_input)
        render_block("AGENT", resp.text, AGENT_COLOR)

#### Testing the prompt

In [6]:
planner_agent(model=MODEL)